# Figure A1 — weight ablation, keep-only vs aligned vs rescaled (random label)

Each panel keeps ONE component of the random-label donor and resets the rest to the upstream
epoch-0 init (seed 21). Solid = keep-only (no align); dashed open marker = keep-only + Procrustes
alignment to the pristine donor; the orange rescaled line keeps direction but scales the kept
weights back to their init Frobenius norm. Slope-1 guide under equal aspect (random label has a
linear-transfer regime); red dashed = from-scratch baseline. All t0 from `common.fit_t0`.

- **Consumes** (ingested metrics):
  `surgery_svd_t0_labelseed42/randlabel/original`,
  `surgery_initreset_labelseed42/randlabel/keep_{bn,fc,conv}`,
  `surgery_align_labelseed42/randlabel/keep_{bn,fc,conv}_aligned`,
  `surgery_rescale_init_labelseed42/randlabel/{fc,conv}_rescale`
- **Produces:** `outputs/figA1_weight_ablation_aligned.png`


In [ ]:
import sys
import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# repo root (dir holding common/) onto sys.path, from any working directory
_root = Path.cwd()
while not (_root / "common").is_dir() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

from common.config import FIGURE_DATA_DIR, OUTPUT_DIR
from common.fit_t0 import fit_t0

UPSTREAM = "randlabel"
GUIDE = True
KGRID = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]

SV  = FIGURE_DATA_DIR / "surgery_svd_t0_labelseed42" / UPSTREAM       # Original reference
AB  = FIGURE_DATA_DIR / "surgery_initreset_labelseed42" / UPSTREAM    # keep-only, non-kept -> epoch-0 init
ABA = FIGURE_DATA_DIR / "surgery_align_labelseed42" / UPSTREAM        # keep-only + Procrustes align
FR  = FIGURE_DATA_DIR / "surgery_rescale_init_labelseed42" / UPSTREAM / "fc_rescale"
CR  = FIGURE_DATA_DIR / "surgery_rescale_init_labelseed42" / UPSTREAM / "conv_rescale"


def t0_of(metrics_path):
    m = json.loads(Path(metrics_path).read_text(encoding="utf-8"))
    return fit_t0([r["epoch"] for r in m["epochs"]], [r["train_acc"] for r in m["epochs"]]).t0


def curve(cond_dir):
    return np.array([t0_of(cond_dir / f"epoch{k:04d}" / "metrics.json") for k in KGRID], float)

In [ ]:
C_ORIG, C_KEEP, C_RESC = "#1a1a1a", "#2166ac", "#e66101"
FS_LABEL, FS_TICK, FS_LEG, FS_TITLE = 12, 10, 8, 12
# (label, dir, color, style, open_marker) -- solid=keep-only(no align), dashed+open=aligned
PANELS = [
    ("(a) Batch Normalization (BN)", [
        ("Original", SV / "original", C_ORIG, "-o", False),
        ("keep BN only", AB / "keep_bn", C_KEEP, "-s", False),
        ("keep BN only (aligned)", ABA / "keep_bn_aligned", C_KEEP, "--s", True)]),
    ("(b) Fully connected (FC)", [
        ("Original", SV / "original", C_ORIG, "-o", False),
        ("keep FC only", AB / "keep_fc", C_KEEP, "-s", False),
        ("keep FC only (aligned)", ABA / "keep_fc_aligned", C_KEEP, "--s", True),
        ("keep FC rescaled", FR, C_RESC, "-^", False)]),
    ("(c) Convolutional weights", [
        ("Original", SV / "original", C_ORIG, "-o", False),
        ("keep conv only", AB / "keep_conv", C_KEEP, "-s", False),
        ("keep conv only (aligned)", ABA / "keep_conv_aligned", C_KEEP, "--s", True),
        ("keep conv rescaled", CR, C_RESC, "-^", False)]),
]
data = [(title, [(lab, curve(d), col, st, o) for lab, d, col, st, o in lines]) for title, lines in PANELS]

allv = [v for _, lines in data for _, arr, _, _, _ in lines for v in arr]
ylo = 0.0 if GUIDE else math.floor(min(allv) / 5) * 5
yhi = math.ceil(max(allv) / 5) * 5
xlo, xhi = -2.0, max(KGRID) + 2.0
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.7), sharey=True, constrained_layout=True)
orig0 = data[0][1][0][1][0]
for ax, (title, lines) in zip(axes, data):
    ax.plot([0.0, xhi], [orig0, orig0], ls="--", color="#c44e52", lw=1.2, zorder=1,
            label="From-scratch baseline" if ax is axes[0] else None)
    if GUIDE:
        gx = np.array([0.0, orig0])
        ax.plot(gx, orig0 - gx, ls="--", color="0.45", lw=1.3, zorder=1,
                label="slope = \u22121" if ax is axes[0] else None)
    for lab, arr, col, st, opn in lines:
        mk = dict(mec=col, mew=1.1, ms=6, mfc="white") if opn else dict(mec="white", mew=0.5, ms=6)
        ax.plot(KGRID, arr, st, color=col, alpha=0.9, label=lab, **mk)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Pre-training epoch", fontsize=FS_LABEL)
    ax.set_title(title, fontsize=FS_TITLE, pad=8)
    ax.tick_params(labelsize=FS_TICK)
    ax.set_xticks(KGRID if len(KGRID) <= 6 else KGRID[::2])
    ax.set_xlim(xlo, xhi)
    ax.set_ylim(ylo, yhi)
    if GUIDE:
        ax.set_aspect("equal", adjustable="box")
    ax.legend(fontsize=FS_LEG, frameon=False, loc="lower left", handlelength=1.6, borderaxespad=0.4)
axes[0].set_ylabel("$t_0$ (downstream epochs)", fontsize=FS_LABEL)

OUT_PNG = OUTPUT_DIR / "figA1_weight_ablation_aligned.png"
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=400, bbox_inches="tight")
print("saved", OUT_PNG)